## DSAN 6000 Homework 3A: Allocating Tasks to Parallel Workers with `joblib`

## Overview

You made it to the first DSAN 6000 homework introducing a new coding concept! The goal of this part is for you to gain hands-on experience using **`joblib`** to quickly parallelize an **embarrassingly-parallel** task.

In this case, the problem is one you've likely already seen before, the basic **data cleaning** task of "normalizing" the format of a bunch of **phone numbers** that have been submitted by different users of a web form. This basic task was chosen specifically so that you can take your intuitions about how long it might take for a **serial** algorithm to complete, and compare with how quickly you'll be able to complete it by using `joblib` to **distribute** the subtasks to different **workers**, who can process the numbers in parallel.

In [1]:
import boto3

In [4]:
s3 = boto3.client('s3')

In [6]:
s3.download_file('dsan6000-data', 'form_submissions.parquet', 'data/form_submissions.parquet')

In [8]:
import pandas as pd

In [11]:
pnum_df = pd.read_parquet("data/form_submissions.parquet")

In [12]:
pnum_df

,submission_id,submitted,phone_number
0,0,2026-09-15 16:26:29.971077,922-807-4647
1,1,2026-08-25 11:40:00.031323,935-252-1889
2,2,2026-09-01 04:03:40.885958,001-462-593-8264
3,3,2026-09-05 22:38:21.429433,001-754-207-1794
4,4,2026-08-21 21:29:53.281333,549.833.5517
...,...,...,...
999995,999995,2026-09-14 17:58:08.325629,(255)410-5300
999996,999996,2026-08-19 12:26:16.699247,909.241.7423
999997,999997,2026-09-13 21:38:54.245236,674-239-2554
999998,999998,2026-09-14 12:33:10.004583,(879)867-5752


In [13]:
import re

In [25]:
def clean_pnum(pnum):
  return ''.join(re.findall(r'\d', pnum)[-10:])

In [26]:
clean_pnum('+1(281)-330-8004')

'2813308004'

In [34]:
pnums = pnum_df['phone_number'].to_list()
pnums[:20]

['922-807-4647',
 '935-252-1889',
 '001-462-593-8264',
 '001-754-207-1794',
 '549.833.5517',
 '719.328.9783',
 '402.829.6068',
 '+1-848-498-3212',
 '536.634.7380',
 '4192158342',
 '477-388-5167',
 '+1-993-549-3762',
 '001-542-867-9882',
 '584.948.0832',
 '001-308-800-0863',
 '372.205.9058',
 '830-725-8259',
 '001-269-465-5145',
 '651-765-6526',
 '9725205954']

In [45]:
import time
disp_time = lambda start, end: print('{:.4f} s'.format(end - start))

In [46]:
serial_start = time.time()
pnums_cleaned_serial = [clean_pnum(p) for p in pnums]
serial_end = time.time()
disp_time(serial_start, serial_end)

2.1338 s


In [37]:
import time

In [31]:
import joblib

In [43]:
parallel_runner = joblib.Parallel()

In [44]:
par_start = time.time()
pnums_cleaned_parallel = parallel_runner(
  joblib.delayed(clean_pnum)(p) for p in pnums
)
par_end = time.time()
disp_time(par_start, par_end)

4.5032 s
